## Calculate kick factors for each IVU gap

The goal of this notebook is to load the elements encoding the IVU geometric and RW impedances for different gaps, and calculate the corresponding theoretical kick factors

### Imports

In [7]:
%matplotlib inline
# %matplotlib notebook
# %matplotlib qt5

from IPython.display import Latex
from IPython.display import SVG

import re
import os
import sh
from functools import reduce
import operator as opr
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot   as mplt
import matplotlib.gridspec as gridspec
import matplotlib.cm       as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable

mplt.rcParams.update(
    {'font.size': 10, 'lines.linewidth':2})

import pyaccel
from pymodels import si as si_lat

import pycolleff.impedances as imp
from pycolleff.impedances import element_and_budget as elbud_mod
import pycolleff.rings.sirius as si
import pycolleff.colleff as colleff

utils = importlib.import_module('utils')

### Create Ring Model

In [ ]:
mod = si_lat.create_accelerator()

Change RF voltage

In [32]:
famdata = si_lat.get_family_data(mod)
# famdata.keys()
pyaccel.lattice.find_indices(mod, 'fam_name', 'SRFCav')
print(mod[677])

fam_name   : SRFCav 
pass_method: cavity_pass 
frequency  : 499663831.62944365 Hz
voltage    : 3000000.0 V
vchamber   : 2 
hmin       : -0.012 m
hmax       : 0.012 m
vmin       : -0.012 m
vmax       : 0.012 m


0.0

In [12]:
ring = si.create_ring()
ring.gap_voltage = 2.1e6
si.update_from_pymodels(ring, mod)
ring.bunlen = 5e-3

print(ring)

Lattice Version             :   SI.v25.01-s05.02  
Circumference [m]           :       518.387       
Revolution Period [us]      :        1.729        
Revolution Frequency [kHz]  :       578.318       
Energy [GeV]                :        3.000        
U0 [keV]                    :       474.890       
Vgap [MV]                   :        2.100        
Momentum Compaction         :       1.64e-04      
Harmonic Number             :         864         
Current [mA]                :       100.000       
Current per Bunch [mA]      :        0.116        
Synchrotron Tune            :       0.00392       
Tunes x/y                   :    49.096/14.152    
Chromaticities x/y          :     2.500/2.500     
Damping Times x/y/e [ms]    :   16.8/ 21.8 /12.8  
Energy Spread [%]           :        0.0851       
Bunch Length [mm]           :        5.000        



Set fractional parts of the x and y tunes to zero

In [13]:
ring.tunex = 49.00
ring.tuney = 14.00

### Filling Patterns

In [14]:
# Experiment filling pattern
fill_pattern1 = np.zeros(864,)
fill_pattern1[10] = 0.9
fill_pattern1[463] = 0.1

In [15]:
# Single bunch
# Equivalent to setting ring.num_bun = 1
fill_pattern2 = np.zeros(864,)
fill_pattern2[431] = 1.0

In [16]:
# Naming them for later use
fill_patterns = {
    "Experiment": fill_pattern1,
    "Single Bunch": fill_pattern2,
}

### Load Elements containing Impedance information

In [17]:
folder = 'elements/'
files = [
    'ivu_flexible_taper_gap_4.pickle',
    'ivu_flexible_taper_gap_14.pickle',
    'ivu_flexible_taper_gap_24.pickle',
    'ivu_rw_gap_4.pickle',
    'ivu_rw_gap_4_extra_vac_layer.pickle',
    'ivu_rw_gap_4_no_magnet.pickle',
    'ivu_rw_gap_14.pickle',
    'ivu_rw_gap_14_extra_vac_layer.pickle',
    'ivu_rw_gap_24.pickle'
]

els = [imp.load_element(folder + file) for file in files]


In [18]:
for el in els:
    print(el.ang_freq[-1]*1e-9 / (2*np.pi))

47.21731213500001
47.21731213500001
47.21731213500001
8800187.954916894
8800187.954916894
8800187.954916894
2514339.4156905413
2514339.4156905413
1466697.9924861484


### Truncate frequencies between +-100 GHz

In [19]:
def truncate_element(el, max_ang_freq):
    if el.ang_freq[-1] < max_ang_freq:  # No need to truncate
        return el
    
    mask = np.abs(el.ang_freq) < max_ang_freq

    el.ang_freq = el.ang_freq[mask]  # Truncate freq

    for prop in ["Zll", "Zdx", "Zdy", "Zqx", "Zqy"]:  # Truncate props
        imp = getattr(el, prop)
        if imp is not None and len(imp) > 0:
            setattr(el, prop, imp[mask])

    return el

In [20]:
max_ang_freq = 2*np.pi*100e9  # Corresponding to 100 GHz 

el_procs = [el.copy() for el in els]

for el_proc in el_procs:
    truncate_element(el_proc, max_ang_freq)

### Calculate Kick Factors

In [21]:
for el in els:
    print(el.name)

IVU Flexible Taper Gap 4
IVU Flexible Taper Gap 14
IVU Flexible Taper Gap 24
IVU RW Gap 4
IVU RW Gap 4 Extra Vac Layer
IVU RW Gap 4 No Magnet
IVU RW Gap 14
IVU RW Gap 14 Extra Vac Layer
IVU RW Gap 24


In [22]:
imp_types = ['Zdy', 'Zqy', 'Zdx', 'Zqx']

results = []

for el in els:

    for fill_name, fill_pattern in fill_patterns.items():

        for imp_type in imp_types:

            # Plane y
            kd_y, _ = ring.kick_factor(
                element=el, imp_type="Zdy", fillpattern=fill_pattern)
            kq_y, _ = ring.kick_factor(
                element=el, imp_type="Zqy", fillpattern=fill_pattern)

            # Plane x
            kd_x, _ = ring.kick_factor(
                element=el, imp_type="Zdx", fillpattern=fill_pattern)
            kq_x, _ = ring.kick_factor(
                element=el, imp_type="Zqx", fillpattern=fill_pattern)

        results.append({
            "name": el.name,
            "fill_pattern": fill_name,
            "kd_y [V/(pC.m)]": 1e-12 * kd_y,
            "kq_y [V/(pC.m)]": 1e-12 * kq_y,
            "k_y [V/(pC.m)]": 1e-12 * (kd_y + kq_y),
            "kd_x [V/(pC.m)]": 1e-12 * kd_x,
            "kq_x [V/(pC.m)]": 1e-12 * kq_x,
            "k_x [V/(pC.m)]": 1e-12 * (kd_x + kq_x)
        })

results = pd.DataFrame(results)

In [23]:
results

,name,fill_pattern,kd_y [V/(pC.m)],kq_y [V/(pC.m)],k_y [V/(pC.m)],kd_x [V/(pC.m)],kq_x [V/(pC.m)],k_x [V/(pC.m)]
0,IVU Flexible Taper Gap 4,Experiment,-298.302819,-19.415690,-317.718510,-37.582396,18.554945,-19.027451
1,IVU Flexible Taper Gap 4,Single Bunch,-298.302802,-19.415690,-317.718492,-37.582396,18.554945,-19.027451
2,IVU Flexible Taper Gap 14,Experiment,-28.479211,-4.786827,-33.266037,-9.251493,4.573923,-4.677570
3,IVU Flexible Taper Gap 14,Single Bunch,-28.479212,-4.786827,-33.266039,-9.251493,4.573923,-4.677570
4,IVU Flexible Taper Gap 24,Experiment,-29.222689,-4.648374,-33.871063,-8.994329,4.325718,-4.668611
5,IVU Flexible Taper Gap 24,Single Bunch,-29.222690,-4.648374,-33.871064,-8.994329,4.325718,-4.668611
6,IVU RW Gap 4,Experiment,-309.618790,-165.664185,-475.282975,-165.664185,165.664185,0.000000
7,IVU RW Gap 4,Single Bunch,-305.136804,-161.548538,-466.685341,-161.548537,161.548537,0.000000
8,IVU RW Gap 4 Extra Vac Layer,Experiment,-309.470474,-165.391798,-474.862271,-165.391797,165.391797,0.000000
9,IVU RW Gap 4 Extra Vac Layer,Single Bunch,-305.015556,-161.325529,-466.341085,-161.325529,161.325529,0.000000


Filter results to keep only essential

In [24]:
idxs = np.array([0, 2, 4, 6, 12, 16])
filtered_results = results.iloc[idxs]
filtered_results = filtered_results.reset_index(drop=True)

k_cols = [
    "k_y [V/(pC.m)]", "kd_y [V/(pC.m)]", "kq_y [V/(pC.m)]",
    "k_x [V/(pC.m)]", "kd_x [V/(pC.m)]", "kq_x [V/(pC.m)]"
]
filtered_results[k_cols] = filtered_results[k_cols].round(3)

filtered_results = filtered_results[[
    "name", "fill_pattern",
    *k_cols
]]

Add total (Geom + RW) kick factors

In [25]:
gaps = [4, 14, 24]
for i in range(3):
    name = f'IVU Total Gap {gaps[i]}'
    row1 = filtered_results.iloc[i]
    row2 = filtered_results.iloc[i+3]
    total_i = {
            "name": name,
            "fill_pattern": 'Experiment',
            "k_y [V/(pC.m)]": (row1['k_y [V/(pC.m)]'] + row2['k_y [V/(pC.m)]']).round(3),
            "kd_y [V/(pC.m)]": (row1['kd_y [V/(pC.m)]'] + row2['kd_y [V/(pC.m)]']).round(3),
            "kq_y [V/(pC.m)]": (row1['kq_y [V/(pC.m)]'] + row2['kq_y [V/(pC.m)]']).round(3),
            "k_x [V/(pC.m)]": (row1['k_x [V/(pC.m)]'] + row2['k_x [V/(pC.m)]']).round(3),
            "kd_x [V/(pC.m)]": (row1['kd_x [V/(pC.m)]'] + row2['kd_x [V/(pC.m)]']).round(3),
            "kq_x [V/(pC.m)]": (row1['kq_x [V/(pC.m)]'] + row2['kq_x [V/(pC.m)]']).round(3),
        }
    filtered_results.loc[len(filtered_results)] = total_i


In [26]:
filtered_results

,name,fill_pattern,k_y [V/(pC.m)],kd_y [V/(pC.m)],kq_y [V/(pC.m)],k_x [V/(pC.m)],kd_x [V/(pC.m)],kq_x [V/(pC.m)]
0,IVU Flexible Taper Gap 4,Experiment,-317.719,-298.303,-19.416,-19.027,-37.582,18.555
1,IVU Flexible Taper Gap 14,Experiment,-33.266,-28.479,-4.787,-4.678,-9.251,4.574
2,IVU Flexible Taper Gap 24,Experiment,-33.871,-29.223,-4.648,-4.669,-8.994,4.326
3,IVU RW Gap 4,Experiment,-475.283,-309.619,-165.664,0.000,-165.664,165.664
4,IVU RW Gap 14,Experiment,-13.972,-8.712,-5.260,0.000,-5.260,5.260
5,IVU RW Gap 24,Experiment,-3.346,-2.027,-1.319,0.000,-1.319,1.319
6,IVU Total Gap 4,Experiment,-793.002,-607.922,-185.080,-19.027,-203.246,184.219
7,IVU Total Gap 14,Experiment,-47.238,-37.191,-10.047,-4.678,-14.511,9.834
8,IVU Total Gap 24,Experiment,-37.217,-31.250,-5.967,-4.669,-10.313,5.645


In [31]:
(4/4.3)**3 * 793.002

638.3353415422541

In [27]:
slides_table = filtered_results.loc[range(9), ['name', 'k_y [V/(pC.m)]', 'k_x [V/(pC.m)]']]
slides_table

,name,k_y [V/(pC.m)],k_x [V/(pC.m)]
0,IVU Flexible Taper Gap 4,-317.719,-19.027
1,IVU Flexible Taper Gap 14,-33.266,-4.678
2,IVU Flexible Taper Gap 24,-33.871,-4.669
3,IVU RW Gap 4,-475.283,0.000
4,IVU RW Gap 14,-13.972,0.000
5,IVU RW Gap 24,-3.346,0.000
6,IVU Total Gap 4,-793.002,-19.027
7,IVU Total Gap 14,-47.238,-4.678
8,IVU Total Gap 24,-37.217,-4.669
